# ARC-v0.37.2.2 — HotpotQA-GTE Native-Gold Downstream QA Consequence Audit
## Frozen execution notebook

This notebook executes the prospectively frozen ARC-v0.37.1 benchmark.

### Frozen upstream facts

The executed ARC-v0.37.1 freeze established:

- HotpotQA native-gold alignment: **7,405 / 7,405**
- FIT: **3,702**
- VALID: **3,703**
- MAIN: **500 untouched VALID queries**
- FIT SHA256: `cbe2b26a6f951c846d33fde0582937335b6c63b483ee394d6f3304a5665c28c4`
- VALID SHA256: `0d4de4bec80bc7a6dee653055dfe4c32f76dad655af64ea3dc1c55024b36f085`
- MAIN SHA256: `7506c81ce5160cd329400240e17bcc61a92317f6e3f6a8f814c6f0105790b151`
- executed ARC-v0.37.1 protocol SHA256:
  `6177213b17f933ac99601e6eea4a7cd657701e2cc3fb3e1c7d83d0997167c08a`

### Scientific choices carried forward unchanged

- BEIR HotpotQA global corpus
- native official HotpotQA answer field
- `thenlper/gte-small`, 384 dimensions, normalized embeddings
- IVF-PQ32 @ nprobe=64 as representation-low
- IVF-SQ8 @ nprobe=64 as shared high
- IVF-SQ8 @ FIT-selected nprobe as search-effort-low
- search grid `[1,2,4,8,16,32]`
- one-shot calibration metric: nDCG@10
- anchored centroid, H=4
- 8 frozen policies: alpha `{0.1,0.3,0.5,0.7}` × `{mean-k20, softmax-k20-tau0.1}`
- Qwen2.5-3B-Instruct, deterministic decoding
- terminal top-5 passages, title + text, max 850 characters/passage
- primary answer metric: normalized token F1
- primary estimand:
  `mean_q[(mean_policy F1_search,H - F1_rep,H) - (F1_search,0 - F1_rep,0)]`
- query is the independent sampling unit
- 10,000 paired-query bootstrap replicates
- positive, null, and reversed outcomes are all retained

### Additional reproducibility freeze in v0.37.2

Before any effectiveness outcome is computed, this notebook resolves and pins the exact
Hugging Face revisions for the encoder and answerer and byte-freezes the answer prompt
contract. These are implementation/reproducibility specifications, not outcome-driven
changes to the scientific design.

The notebook is deliberately resumable. Large corpus embeddings and FAISS indexes are
cached on Google Drive.

### v0.37.2.1 engineering-only repair

ARC-v0.37.2 stopped during FIT-only calibration, before an nprobe was selected
and before any MAIN500 retrieval or answer outcome existed. FAISS IVF search may
legitimately pad top-k results with row ID -1 when the probed inverted lists contain
fewer than k candidates. The previous helper incorrectly rejected any such padding.

This repair changes only padding handling. Padded IDs are treated as absent
candidates for utility/Jaccard; the frozen feedback k=20 and answer evidence k=5
remain hard requirements. Search grid, calibration criterion, membership, ANN
mechanisms, H, policies, models, prompt, metrics, and primary estimand are unchanged.


### v0.37.2.2 engineering-only scope repair

v0.37.2.1 stopped again inside FIT-only calibration before nprobe selection and before
any MAIN500 or answer outcome. The new padding helper existed in the updated retrieval
helper cell, but the active Colab runtime still held the earlier helper definitions, so
`padding_diagnostics` was not in scope when the calibration cell was rerun.

v0.37.2.2 makes the calibration cell self-contained by defining its local padding
diagnostic function inside the cell. No scientific choice changes.


In [ ]:
# Cell 1 — Install pinned/compatible dependencies
import sys, subprocess

def install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

install(
    "faiss-cpu==1.12.0",
    "sentence-transformers>=5.0,<6",
    "transformers>=4.55,<5",
    "huggingface_hub>=0.34",
    "accelerate>=1.5",
    "pyarrow",
    "tqdm",
    "requests",
)
print("DEPENDENCIES — PASS")

In [ ]:
# Cell 2 — Imports, constants, helpers, and persistent Drive root
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from functools import lru_cache
import gc, hashlib, json, math, os, random, re, shutil, string, sys, unicodedata, zipfile

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

SEED = 20260837
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Frozen benchmark / retrieval choices
DIM = 384
NLIST = 4096
PQ_M = 32
PQ_NBITS = 8
HIGH_NPROBE = 64
NPROBE_GRID = [1, 2, 4, 8, 16, 32]
TRAIN_DOCS = 250_000
TOP_RETRIEVE = 100
NDCG_K = 10
H = 4
FEEDBACK_K = 20

# Frozen downstream choices
EVIDENCE_K = 5
MAX_PASSAGE_CHARS = 850
ANSWER_MAX_NEW_TOKENS = 32
ANSWER_BATCH_SIZE = 8
BOOTSTRAP_REPS = 10_000

ENCODER_MODEL = "thenlper/gte-small"
ANSWER_MODEL = "Qwen/Qwen2.5-3B-Instruct"

# Storage / performance
EMBED_BLOCK_ROWS = 50_000
EMBED_BATCH_SIZE = 384
SEARCH_BATCH_SIZE = 256
N_DOCS_EXPECTED = 5_233_329
N_FULL_EXPECTED = 7_405
N_FIT_EXPECTED = 3_702
N_VALID_EXPECTED = 3_703
N_MAIN_EXPECTED = 500

# Frozen v0.37.1 lineage
SPLIT_SALT = "ARC-v0.37-HOTPOTQA-NATIVE-GOLD-SPLIT-v1"
MAIN_SALT = "ARC-v0.37-HOTPOTQA-NATIVE-GOLD-MAIN500-v1"
EXPECTED_BEIR_ZIP_SHA = "c62459dbc91d31329584507b1fa564b3b1abcee58800967113054ee0f5a15c62"
EXPECTED_HOTPOT_SHA = "4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea"
EXPECTED_FIT_SHA = "cbe2b26a6f951c846d33fde0582937335b6c63b483ee394d6f3304a5665c28c4"
EXPECTED_VALID_SHA = "0d4de4bec80bc7a6dee653055dfe4c32f76dad655af64ea3dc1c55024b36f085"
EXPECTED_MAIN_SHA = "7506c81ce5160cd329400240e17bcc61a92317f6e3f6a8f814c6f0105790b151"
SOURCE_V0371_PROTOCOL_SHA = "6177213b17f933ac99601e6eea4a7cd657701e2cc3fb3e1c7d83d0997167c08a"

POLICIES = []
for alpha in [0.1, 0.3, 0.5, 0.7]:
    POLICIES.append({
        "name": f"mean-k20-a{alpha}",
        "family": "mean",
        "k": 20,
        "tau": None,
        "alpha": alpha,
    })
    POLICIES.append({
        "name": f"softmax-k20-t0.1-a{alpha}",
        "family": "softmax",
        "k": 20,
        "tau": 0.1,
        "alpha": alpha,
    })
assert len(POLICIES) == 8

def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def sha256_text(s):
    return hashlib.sha256(str(s).encode("utf-8")).hexdigest()

def membership_sha(ids):
    # Exactly the v0.37.1 convention: sorted IDs + trailing newline.
    return sha256_text("\n".join(sorted(map(str, ids))) + "\n")

def salted_rank(qid, salt):
    return hashlib.sha256(f"{salt}\n{qid}".encode("utf-8")).hexdigest()

def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

def ols_slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    xc = x - x.mean()
    return float(np.dot(xc, y - y.mean()) / np.dot(xc, xc))

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0" / "hotpotqa-gte-native-gold-v0372"
CACHE = ROOT / "cache"
EMB_ROOT = CACHE / "corpus_embeddings_f16"
IDX_ROOT = CACHE / "indexes"
RUN = ROOT / "run-frozen-main500-v1"

for p in [ROOT, CACHE, EMB_ROOT, IDX_ROOT, RUN]:
    p.mkdir(parents=True, exist_ok=True)

print("Persistent root:", ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "GPU runtime required; A100-class GPU strongly recommended."

In [ ]:
# Cell 3 — Acquire BEIR HotpotQA + pinned native-gold HotpotQA dev data
RAW = Path("/content/arc-v0372-hotpot")
RAW.mkdir(parents=True, exist_ok=True)

BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/hotpotqa.zip"
BEIR_ZIP = RAW / "hotpotqa.zip"
BEIR_DIR = RAW / "hotpotqa"
CORPUS_JSONL = BEIR_DIR / "corpus.jsonl"
QUERIES_JSONL = BEIR_DIR / "queries.jsonl"
QRELS_TEST = BEIR_DIR / "qrels" / "test.tsv"

HOTPOT_MIRROR_URL = (
    "https://huggingface.co/datasets/RAGLAB/data/resolve/main/"
    "eval_datasets/HotPotQA/hotpot_dev_distractor_v1.json?download=true"
)
HOTPOT_JSON = RAW / "hotpot_dev_distractor_v1.json"

def download_stream(url, dest, timeout=(30, 600)):
    tmp = Path(str(dest) + ".part")
    if tmp.exists():
        tmp.unlink()
    with requests.get(
        url,
        stream=True,
        timeout=timeout,
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0 ARC-v0.37.2 research execution"},
    ) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp.replace(dest)

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file() and QRELS_TEST.is_file()):
    if not BEIR_ZIP.is_file():
        print("Downloading BEIR HotpotQA...")
        download_stream(BEIR_URL, BEIR_ZIP, timeout=(30, 1200))
    assert sha256_file(BEIR_ZIP) == EXPECTED_BEIR_ZIP_SHA, "BEIR ZIP SHA mismatch."
    print("Extracting BEIR HotpotQA...")
    with zipfile.ZipFile(BEIR_ZIP) as z:
        z.extractall(RAW)

assert sha256_file(BEIR_ZIP) == EXPECTED_BEIR_ZIP_SHA
assert CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file() and QRELS_TEST.is_file()

if not HOTPOT_JSON.is_file():
    print("Downloading pinned HotpotQA native-gold dev file...")
    download_stream(HOTPOT_MIRROR_URL, HOTPOT_JSON, timeout=(30, 600))

assert sha256_file(HOTPOT_JSON) == EXPECTED_HOTPOT_SHA, "HotpotQA native-gold SHA mismatch."

OFFICIAL = json.loads(HOTPOT_JSON.read_text(encoding="utf-8"))
assert isinstance(OFFICIAL, list) and len(OFFICIAL) == N_FULL_EXPECTED
official_by_id = {str(x["_id"]): x for x in OFFICIAL}
assert len(official_by_id) == N_FULL_EXPECTED

print("BEIR SHA:", sha256_file(BEIR_ZIP))
print("HotpotQA SHA:", sha256_file(HOTPOT_JSON))
print("SOURCE BYTES — PASS")

In [ ]:
# Cell 4 — Reconstruct and hard-verify frozen FIT / VALID / MAIN500 membership and native gold
QUERY_TEXT = {}
with open(QUERIES_JSONL, encoding="utf-8") as f:
    for line in f:
        o = json.loads(line)
        QUERY_TEXT[str(o["_id"])] = str(o.get("text", ""))

qrels_df = pd.read_csv(QRELS_TEST, sep="\t")
qc = next(c for c in ["query-id", "query_id", "qid"] if c in qrels_df.columns)
dc = next(c for c in ["corpus-id", "corpus_id", "doc_id"] if c in qrels_df.columns)
sc = next((c for c in ["score", "relevance", "rel"] if c in qrels_df.columns), None)
qrels_df[qc] = qrels_df[qc].astype(str)
qrels_df[dc] = qrels_df[dc].astype(str)
if sc is not None:
    qrels_df = qrels_df[pd.to_numeric(qrels_df[sc], errors="coerce") > 0].copy()

TEST_IDS = sorted(qrels_df[qc].unique().tolist())
OFFICIAL_IDS = sorted(official_by_id)
assert len(TEST_IDS) == len(OFFICIAL_IDS) == N_FULL_EXPECTED
assert set(TEST_IDS) == set(OFFICIAL_IDS)

def norm_question(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

mismatch = [
    q for q in TEST_IDS
    if norm_question(QUERY_TEXT[q]) != norm_question(official_by_id[q]["question"])
]
assert not mismatch, f"Question-text mismatch: {mismatch[:10]}"

ORDERED = sorted(TEST_IDS, key=lambda q: (salted_rank(q, SPLIT_SALT), q))
mid = len(ORDERED) // 2
FIT_IDS = sorted(ORDERED[:mid])
VALID_IDS = sorted(ORDERED[mid:])
MAIN_IDS = sorted(
    sorted(VALID_IDS, key=lambda q: (salted_rank(q, MAIN_SALT), q))[:N_MAIN_EXPECTED]
)

assert len(FIT_IDS) == N_FIT_EXPECTED
assert len(VALID_IDS) == N_VALID_EXPECTED
assert len(MAIN_IDS) == N_MAIN_EXPECTED
assert membership_sha(FIT_IDS) == EXPECTED_FIT_SHA
assert membership_sha(VALID_IDS) == EXPECTED_VALID_SHA
assert membership_sha(MAIN_IDS) == EXPECTED_MAIN_SHA
assert set(MAIN_IDS).issubset(set(VALID_IDS))

GOLD = {q: str(official_by_id[q]["answer"]).strip() for q in MAIN_IDS}
assert all(GOLD.values())

lineage = {
    "study_id": "ARC-v0.37.2.2",
    "source_v0371_protocol_sha256": SOURCE_V0371_PROTOCOL_SHA,
    "beir_zip_sha256": sha256_file(BEIR_ZIP),
    "hotpot_native_gold_sha256": sha256_file(HOTPOT_JSON),
    "n_full": len(TEST_IDS),
    "fit_n": len(FIT_IDS),
    "valid_n": len(VALID_IDS),
    "main_n": len(MAIN_IDS),
    "fit_sha256": membership_sha(FIT_IDS),
    "valid_sha256": membership_sha(VALID_IDS),
    "main_sha256": membership_sha(MAIN_IDS),
    "question_text_mismatches": 0,
}
(RUN / "V0372_LINEAGE_CHECK.json").write_text(
    json.dumps(lineage, indent=2, sort_keys=True), encoding="utf-8"
)

print("FIT:", len(FIT_IDS), membership_sha(FIT_IDS))
print("VALID:", len(VALID_IDS), membership_sha(VALID_IDS))
print("MAIN:", len(MAIN_IDS), membership_sha(MAIN_IDS))
print("V0.37.1 MEMBERSHIP RECONSTRUCTION — PASS")

In [ ]:
# Cell 5 — Resolve and pin model revisions + byte-freeze prompt contract BEFORE outcomes
ENCODER_REVISION = str(model_info(ENCODER_MODEL).sha)
ANSWER_REVISION = str(model_info(ANSWER_MODEL).sha)

SYSTEM_PROMPT = (
    "You answer open-domain factual questions from retrieved evidence. "
    "Return only the shortest answer span that answers the question. "
    "Do not explain, cite passages, or add extra text. "
    "If the evidence is insufficient, return your best concise answer."
)

USER_TEMPLATE = (
    "QUESTION:\n{question}\n\n"
    "RETRIEVED PASSAGES:\n{passages}\n\n"
    "ANSWER:"
)

PROMPT_CONTRACT = {
    "system": SYSTEM_PROMPT,
    "user_template": USER_TEMPLATE,
    "passage_format": "[{rank}] {title}\\n{text}",
    "passage_separator": "\\n\\n",
    "evidence_k": EVIDENCE_K,
    "max_passage_chars": MAX_PASSAGE_CHARS,
    "scores_exposed": False,
}
PROMPT_SHA = sha256_text(json.dumps(PROMPT_CONTRACT, sort_keys=True, ensure_ascii=False))

RUNTIME_PROTOCOL = {
    "study_id": "ARC-v0.37.2.2",
    "status": "RUNTIME_FROZEN_BEFORE_EFFECTIVENESS_OUTCOMES",
    "engineering_repair": {
        "parent": "ARC-v0.37.2",
        "failure_stage": "FIT-only calibration before nprobe selection",
        "failure": "FAISS IVF top-k returned legal -1 padding",
        "repair": "ignore -1 as absent candidate; preserve hard k20 feedback and k5 evidence requirements",
        "selected_nprobe_seen_before_fix": False,
        "main_outcomes_seen_before_fix": False,
        "answer_outcomes_seen_before_fix": False,
        "scientific_choices_changed": False,
        "v03722_scope_fix": "calibration-local padding diagnostics to avoid stale Colab helper scope"
    },
    "source_freeze": {
        "v0371_protocol_sha256": SOURCE_V0371_PROTOCOL_SHA,
        "fit_sha256": EXPECTED_FIT_SHA,
        "valid_sha256": EXPECTED_VALID_SHA,
        "main_sha256": EXPECTED_MAIN_SHA,
    },
    "dataset": {
        "retrieval": "BEIR HotpotQA global corpus",
        "qa_gold": "HotpotQA dev distractor native answer field",
        "n_full": N_FULL_EXPECTED,
        "n_fit": N_FIT_EXPECTED,
        "n_valid": N_VALID_EXPECTED,
        "n_main": N_MAIN_EXPECTED,
    },
    "encoder": {
        "model": ENCODER_MODEL,
        "revision": ENCODER_REVISION,
        "dimension": DIM,
        "normalize_embeddings": True,
        "text_serialization": "title + single space + text",
    },
    "ann": {
        "nlist": NLIST,
        "pq_m": PQ_M,
        "pq_nbits": PQ_NBITS,
        "train_docs": TRAIN_DOCS,
        "representation_low": f"IVF-PQ{PQ_M}@nprobe{HIGH_NPROBE}",
        "shared_high": f"IVF-SQ8@nprobe{HIGH_NPROBE}",
        "search_effort_low": "IVF-SQ8@FIT-selected-nprobe",
        "search_grid": NPROBE_GRID,
        "calibration": (
            "FIT-only minimum absolute mismatch to mean representation one-shot nDCG@10 loss; "
            "ties choose smaller nprobe"
        ),
        "top_retrieve": TOP_RETRIEVE,
    },
    "feedback": {
        "operator": "anchored centroid",
        "H": H,
        "feedback_k": FEEDBACK_K,
        "policies": POLICIES,
    },
    "answerer": {
        "model": ANSWER_MODEL,
        "revision": ANSWER_REVISION,
        "do_sample": False,
        "max_new_tokens": ANSWER_MAX_NEW_TOKENS,
        "evidence_k": EVIDENCE_K,
        "max_passage_chars": MAX_PASSAGE_CHARS,
        "prompt_contract": PROMPT_CONTRACT,
        "prompt_sha256": PROMPT_SHA,
    },
    "primary": {
        "metric": "HotpotQA-style normalized token F1",
        "D0": "F1_search_low_t0 - F1_rep_low_t0",
        "DH": "mean_policy(F1_search_low_H - F1_rep_low_H)",
        "estimand": "mean_query(DH - D0)",
        "classification": {
            "positive": "95% paired-query bootstrap CI strictly > 0",
            "reversed": "95% paired-query bootstrap CI strictly < 0",
            "unresolved": "95% CI includes 0",
        },
    },
    "secondary_prespecified": [
        "EM analogue of primary",
        "one-shot and terminal F1 search-minus-representation contrasts",
        "terminal high-minus-low F1 contrasts",
        "round-0 and terminal normalized-answer disagreement to shared high",
        "retrieval-side H3_abs representation-minus-search bridge on MAIN500",
    ],
    "statistics": {
        "independent_unit": "query",
        "within_query_policy_aggregation": "mean over frozen 8 policies",
        "bootstrap_reps": BOOTSTRAP_REPS,
        "seed": SEED,
    },
    "retention_rule": (
        "Retain positive, null, or reversed outcomes. "
        "No MAIN membership, model, prompt, evidence depth, horizon, policy, metric, "
        "or primary-estimand tuning after this freeze."
    ),
}

PROTO_PATH = RUN / "V0372_RUNTIME_FROZEN_PROTOCOL.json"
PROTO_PATH.write_text(
    json.dumps(RUNTIME_PROTOCOL, indent=2, sort_keys=True, ensure_ascii=False),
    encoding="utf-8",
)
RUNTIME_PROTOCOL_SHA = sha256_file(PROTO_PATH)
(RUN / "V0372_RUNTIME_PROTOCOL_SHA256.txt").write_text(
    RUNTIME_PROTOCOL_SHA + "\n", encoding="utf-8"
)

print("Encoder revision:", ENCODER_REVISION)
print("Answer revision:", ANSWER_REVISION)
print("Prompt SHA:", PROMPT_SHA)
print("Runtime protocol SHA:", RUNTIME_PROTOCOL_SHA)
print("V0.37.2 RUNTIME FREEZE — PASS")

## Provenance checkpoint

At this point no one-shot nDCG calibration, MAIN retrieval trajectory, or answer output
has been computed by ARC-v0.37.2.

If you want a maximal Git trail, save this notebook now as the frozen execution source
before running the embedding / calibration cells. Do not alter the scientific choices below.

In [ ]:
# Cell 6 — Build corpus byte offsets and relevant-doc row mapping without a 5.23M-ID Python dictionary
OFFSETS_PATH = CACHE / "hotpot_corpus_offsets.npy"
QREL_ROW_MAP_PATH = CACHE / "hotpot_test_qrel_doc_rows.json"

# Native qrels in document-ID space.
REL_DOCIDS = defaultdict(set)
for r in qrels_df.itertuples(index=False):
    qid = str(getattr(r, qc.replace("-", "_"), None)) if False else None

# Avoid namedtuple column-name transformations by using DataFrame columns directly.
for _, r in qrels_df.iterrows():
    REL_DOCIDS[str(r[qc])].add(str(r[dc]))

all_relevant_docids = set()
for ds in REL_DOCIDS.values():
    all_relevant_docids.update(ds)

need_scan = (not OFFSETS_PATH.is_file()) or (not QREL_ROW_MAP_PATH.is_file())
if need_scan:
    print("Scanning 5.23M corpus once for byte offsets + qrel row mapping...")
    offsets_mm = np.lib.format.open_memmap(
        OFFSETS_PATH,
        mode="w+",
        dtype=np.int64,
        shape=(N_DOCS_EXPECTED,),
    )
    found = {}
    n = 0
    with open(CORPUS_JSONL, "rb") as f:
        while True:
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            if n >= N_DOCS_EXPECTED:
                raise RuntimeError("Corpus has more rows than frozen expected count.")
            offsets_mm[n] = pos
            o = json.loads(line)
            did = str(o["_id"])
            if did in all_relevant_docids:
                found[did] = n
            n += 1
    offsets_mm.flush()
    del offsets_mm

    assert n == N_DOCS_EXPECTED, (n, N_DOCS_EXPECTED)
    missing_rel = sorted(all_relevant_docids - set(found))
    assert not missing_rel, f"Relevant documents missing from corpus: {missing_rel[:20]}"
    QREL_ROW_MAP_PATH.write_text(json.dumps(found), encoding="utf-8")

OFFSETS = np.load(OFFSETS_PATH, mmap_mode="r")
assert len(OFFSETS) == N_DOCS_EXPECTED
DOCID_TO_ROW_RELEVANT = {
    str(k): int(v)
    for k, v in json.loads(QREL_ROW_MAP_PATH.read_text(encoding="utf-8")).items()
}

RELS = {}
for qid in TEST_IDS:
    RELS[qid] = {
        DOCID_TO_ROW_RELEVANT[d]
        for d in REL_DOCIDS[qid]
        if d in DOCID_TO_ROW_RELEVANT
    }
assert all(RELS[q] for q in TEST_IDS)

corpus_fh = open(CORPUS_JSONL, "rb")

def corpus_row(row):
    row = int(row)
    assert 0 <= row < N_DOCS_EXPECTED
    corpus_fh.seek(int(OFFSETS[row]))
    return json.loads(corpus_fh.readline())

def serialize_doc(o):
    title = str(o.get("title", "") or "").strip()
    body = str(o.get("text", "") or "").replace("\n", " ").strip()
    return (title + " " + body).strip() if title else body

def evidence_for_rows(rows):
    valid_rows = [int(x) for x in rows if int(x) >= 0]
    assert len(valid_rows) >= EVIDENCE_K, (
        f"Fewer than frozen evidence_k={EVIDENCE_K} real candidates."
    )
    out = []
    for row in valid_rows[:EVIDENCE_K]:
        o = corpus_row(row)
        out.append({
            "row": row,
            "doc_id": str(o["_id"]),
            "title": str(o.get("title", "") or "").strip(),
            "text": str(o.get("text", "") or "").replace("\n", " ").strip()[:MAX_PASSAGE_CHARS],
        })
    return out

print("Corpus rows:", len(OFFSETS))
print("Relevant docs mapped:", len(DOCID_TO_ROW_RELEVANT))
print("CORPUS ROW SEMANTICS — PASS")

In [ ]:
# Cell 7 — Load pinned GTE-small encoder
encoder = SentenceTransformer(
    ENCODER_MODEL,
    revision=ENCODER_REVISION,
    device="cuda",
)
actual_dim = int(encoder.get_sentence_embedding_dimension())
assert actual_dim == DIM, (actual_dim, DIM)
print("Encoder:", ENCODER_MODEL, ENCODER_REVISION)
print("Dimension:", actual_dim)
print("ENCODER LOAD — PASS")

In [ ]:
# Cell 8 — Encode 5.23M corpus to resumable float16 blocks on Drive
N_BLOCKS = math.ceil(N_DOCS_EXPECTED / EMBED_BLOCK_ROWS)

def block_path(b, s, e):
    return EMB_ROOT / f"block_{b:04d}_{s}_{e}.npy"

# Sequential reading is much faster than 5.23M random seeks.
with open(CORPUS_JSONL, "rb") as f:
    row = 0
    for b in range(N_BLOCKS):
        s = b * EMBED_BLOCK_ROWS
        e = min(N_DOCS_EXPECTED, (b + 1) * EMBED_BLOCK_ROWS)
        assert row == s

        path = block_path(b, s, e)
        valid_existing = False
        if path.is_file():
            try:
                a = np.load(path, mmap_mode="r")
                valid_existing = (a.shape == (e - s, DIM) and a.dtype == np.float16)
                del a
            except Exception:
                valid_existing = False

        texts = None if valid_existing else []
        for _ in range(s, e):
            line = f.readline()
            if not line:
                raise RuntimeError(f"Unexpected EOF while reading block {b}.")
            if texts is not None:
                texts.append(serialize_doc(json.loads(line)))
            row += 1

        if valid_existing:
            print(f"[{b+1}/{N_BLOCKS}] skip cached {path.name}")
            continue

        print(f"[{b+1}/{N_BLOCKS}] encoding rows {s}:{e}")
        vec = encoder.encode(
            texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        ).astype(np.float16)

        assert vec.shape == (e - s, DIM)
        tmp = Path(str(path) + ".tmp.npy")
        np.save(tmp, vec)
        tmp.replace(path)

        del texts, vec
        gc.collect()
        torch.cuda.empty_cache()

assert row == N_DOCS_EXPECTED

blocks = []
for b in range(N_BLOCKS):
    s = b * EMBED_BLOCK_ROWS
    e = min(N_DOCS_EXPECTED, (b + 1) * EMBED_BLOCK_ROWS)
    p = block_path(b, s, e)
    a = np.load(p, mmap_mode="r")
    assert a.shape == (e - s, DIM) and a.dtype == np.float16
    blocks.append({
        "block": b,
        "start": s,
        "end": e,
        "path": str(p),
        "bytes": p.stat().st_size,
    })
    del a

EMBED_MANIFEST = {
    "status": "COMPLETE",
    "study_id": "ARC-v0.37.2.2",
    "encoder": ENCODER_MODEL,
    "encoder_revision": ENCODER_REVISION,
    "dimension": DIM,
    "corpus_rows": N_DOCS_EXPECTED,
    "block_rows": EMBED_BLOCK_ROWS,
    "dtype": "float16",
    "normalized_before_float16_persistence": True,
    "blocks": blocks,
}
EMBED_MANIFEST_PATH = CACHE / "V0372_CORPUS_EMBEDDING_MANIFEST.json"
EMBED_MANIFEST_PATH.write_text(
    json.dumps(EMBED_MANIFEST, indent=2, sort_keys=True), encoding="utf-8"
)

print("Corpus embedding blocks:", len(blocks))
print("CORPUS EMBEDDINGS — COMPLETE")

In [ ]:
# Cell 9 — Load embedding store, encode all 7,405 queries, and freeze train rows
block_arrays = {}
for rec in EMBED_MANIFEST["blocks"]:
    b = int(rec["block"])
    block_arrays[b] = np.load(rec["path"], mmap_mode="r")

def gather_vectors(rows):
    rows = np.asarray(rows, dtype=np.int64)
    shape = rows.shape
    flat = rows.reshape(-1)
    if np.any(flat < 0) or np.any(flat >= N_DOCS_EXPECTED):
        raise ValueError("Invalid corpus row.")
    out = np.empty((len(flat), DIM), dtype=np.float32)
    bids = flat // EMBED_BLOCK_ROWS
    for b in np.unique(bids):
        mask = bids == b
        local = flat[mask] - int(b) * EMBED_BLOCK_ROWS
        out[mask] = np.asarray(block_arrays[int(b)][local], dtype=np.float32)
    return normalize_rows(out).reshape(*shape, DIM)

QUERY_IDS_PATH = CACHE / "v0372_query_ids.txt"
QUERY_EMB_PATH = CACHE / "v0372_query_embeddings.float32.npy"

ALL_QUERY_IDS = sorted(TEST_IDS)
if not QUERY_IDS_PATH.is_file():
    QUERY_IDS_PATH.write_text("\n".join(ALL_QUERY_IDS) + "\n", encoding="utf-8")
else:
    assert QUERY_IDS_PATH.read_text(encoding="utf-8").splitlines() == ALL_QUERY_IDS

if not QUERY_EMB_PATH.is_file():
    qv = encoder.encode(
        [QUERY_TEXT[q] for q in ALL_QUERY_IDS],
        batch_size=512,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    assert qv.shape == (N_FULL_EXPECTED, DIM)
    np.save(QUERY_EMB_PATH, qv)
    del qv

QUERY_EMB = np.load(QUERY_EMB_PATH, mmap_mode="r")
assert QUERY_EMB.shape == (N_FULL_EXPECTED, DIM)
QPOS = {q: i for i, q in enumerate(ALL_QUERY_IDS)}

TRAIN_ROWS_PATH = CACHE / "v0372_train_rows.npy"
if not TRAIN_ROWS_PATH.is_file():
    rng = np.random.default_rng(SEED)
    rows = np.sort(
        rng.choice(N_DOCS_EXPECTED, size=min(TRAIN_DOCS, N_DOCS_EXPECTED), replace=False)
    ).astype(np.int64)
    np.save(TRAIN_ROWS_PATH, rows)

TRAIN_ROWS = np.load(TRAIN_ROWS_PATH)
assert len(TRAIN_ROWS) == TRAIN_DOCS
TRAIN = gather_vectors(TRAIN_ROWS)

print("Query embeddings:", QUERY_EMB.shape)
print("Train:", TRAIN.shape)
print("Train rows SHA:", sha256_file(TRAIN_ROWS_PATH))
print("QUERY + TRAIN ARTIFACTS — PASS")

In [ ]:
# Cell 10 — Train/build frozen IVF-PQ32 and IVF-SQ8 indexes
PQ_PATH = IDX_ROOT / "hotpot-gte-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = IDX_ROOT / "hotpot-gte-ivfsq8-nlist4096.faiss"

faiss.omp_set_num_threads(max(1, os.cpu_count() or 1))

def new_pq():
    quantizer = faiss.IndexFlatIP(DIM)
    idx = faiss.IndexIVFPQ(
        quantizer, DIM, NLIST, PQ_M, PQ_NBITS, faiss.METRIC_INNER_PRODUCT
    )
    idx.train(np.ascontiguousarray(TRAIN, dtype=np.float32))
    return idx

def new_sq():
    quantizer = faiss.IndexFlatIP(DIM)
    idx = faiss.IndexIVFScalarQuantizer(
        quantizer,
        DIM,
        NLIST,
        faiss.ScalarQuantizer.QT_8bit,
        faiss.METRIC_INNER_PRODUCT,
    )
    idx.train(np.ascontiguousarray(TRAIN, dtype=np.float32))
    return idx

def load_if_complete(path):
    if not path.is_file():
        return None
    idx = faiss.read_index(str(path))
    if idx.d != DIM or idx.ntotal != N_DOCS_EXPECTED:
        return None
    return idx

pq = load_if_complete(PQ_PATH)
sq = load_if_complete(SQ_PATH)

if pq is None or sq is None:
    print("Building fresh indexes. Embeddings are already cached/resumable.")
    pq = new_pq()
    sq = new_sq()

    for rec in tqdm(EMBED_MANIFEST["blocks"], desc="FAISS add"):
        b = int(rec["block"])
        x = normalize_rows(np.asarray(block_arrays[b], dtype=np.float32))
        pq.add(np.ascontiguousarray(x))
        sq.add(np.ascontiguousarray(x))
        del x
        gc.collect()

    assert pq.ntotal == sq.ntotal == N_DOCS_EXPECTED

    pq_tmp = Path(str(PQ_PATH) + ".tmp")
    sq_tmp = Path(str(SQ_PATH) + ".tmp")
    faiss.write_index(pq, str(pq_tmp))
    faiss.write_index(sq, str(sq_tmp))
    pq_tmp.replace(PQ_PATH)
    sq_tmp.replace(SQ_PATH)

assert pq.ntotal == sq.ntotal == N_DOCS_EXPECTED
assert pq.d == sq.d == DIM

INDEX_MANIFEST = {
    "pq_path": str(PQ_PATH),
    "sq_path": str(SQ_PATH),
    "pq_sha256": sha256_file(PQ_PATH),
    "sq_sha256": sha256_file(SQ_PATH),
    "ntotal": int(pq.ntotal),
    "dimension": int(pq.d),
}
(RUN / "V0372_INDEX_MANIFEST.json").write_text(
    json.dumps(INDEX_MANIFEST, indent=2), encoding="utf-8"
)

print("PQ SHA:", INDEX_MANIFEST["pq_sha256"])
print("SQ SHA:", INDEX_MANIFEST["sq_sha256"])
print("INDEX BUILD — PASS")

In [ ]:
# Cell 11 — Retrieval and nDCG@10 helpers
PARAM_SPACE = faiss.ParameterSpace()

def set_nprobe(index, nprobe):
    try:
        index.nprobe = int(nprobe)
    except Exception:
        PARAM_SPACE.set_index_parameter(index, "nprobe", int(nprobe))

def batched_search(index, Q, nprobe, k=TOP_RETRIEVE, batch=SEARCH_BATCH_SIZE):
    set_nprobe(index, nprobe)
    Q = np.ascontiguousarray(Q, dtype=np.float32)
    Ds, Is = [], []
    for s in range(0, len(Q), batch):
        D, I = index.search(Q[s:s+batch], int(k))
        # -1 is legal FAISS padding when fewer than k candidates are available.
        Ds.append(np.asarray(D, dtype=np.float32))
        Is.append(np.asarray(I, dtype=np.int64))
    return np.vstack(Ds), np.vstack(Is)

def ndcg_at_10(rows, rel_rows):
    rows = [int(x) for x in rows[:NDCG_K] if int(x) >= 0]
    dcg = sum(
        1.0 / math.log2(rank + 2)
        for rank, row in enumerate(rows)
        if row in rel_rows
    )
    ideal = min(len(rel_rows), NDCG_K)
    if ideal == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
    return float(dcg / idcg)

def query_matrix(ids):
    pos = np.asarray([QPOS[q] for q in ids], dtype=np.int64)
    return normalize_rows(np.asarray(QUERY_EMB[pos], dtype=np.float32))

def one_shot(index, ids, nprobe):
    Q = query_matrix(ids)
    D, I = batched_search(index, Q, nprobe)
    u = np.asarray(
        [ndcg_at_10(rows, RELS[q]) for q, rows in zip(ids, I)],
        dtype=np.float64,
    )
    return u, D, I

print("RETRIEVAL HELPERS — PASS")

In [ ]:
# Cell 12 — FIT-only nprobe calibration; seal before touching MAIN retrieval outcomes
CALIBRATION_CSV = RUN / "v0372_fit_calibration.csv"

def _fit_padding_diagnostics(I):
    vc = np.sum(np.asarray(I) >= 0, axis=1)
    assert len(vc) > 0
    return {
        "min_valid": int(vc.min()),
        "queries_lt_100": int(np.sum(vc < TOP_RETRIEVE)),
        "queries_lt_feedback_k": int(np.sum(vc < FEEDBACK_K)),
        "queries_lt_evidence_k": int(np.sum(vc < EVIDENCE_K)),
    }

CALIBRATION_GATE = RUN / "v0372_calibration_gate.json"
CALIBRATION_SHA_PATH = RUN / "V0372_CALIBRATION_SHA256.txt"

for _required_name in ["one_shot", "sq", "pq", "FIT_IDS", "HIGH_NPROBE", "NPROBE_GRID"]:
    assert _required_name in globals(), (
        f"Missing runtime dependency `{_required_name}`. "
        "Rerun the prerequisite cells in this notebook before calibration."
    )


if CALIBRATION_GATE.is_file():
    gate = json.loads(CALIBRATION_GATE.read_text(encoding="utf-8"))
    SELECTED_NPROBE = int(gate["selected_nprobe"])
    assert SELECTED_NPROBE in NPROBE_GRID
    print("Reusing sealed FIT calibration:", SELECTED_NPROBE)
else:
    print("Computing FIT-only one-shot calibration...")
    high_fit, _, _ = one_shot(sq, FIT_IDS, HIGH_NPROBE)
    rep_fit, _, _ = one_shot(pq, FIT_IDS, HIGH_NPROBE)
    rep_loss = float(np.mean(high_fit - rep_fit))

    rows = []
    for npb in NPROBE_GRID:
        search_fit, _, search_I = one_shot(sq, FIT_IDS, npb)
        search_loss = float(np.mean(high_fit - search_fit))
        mismatch = abs(search_loss - rep_loss)
        pad = _fit_padding_diagnostics(search_I)
        rows.append({
            "nprobe": int(npb),
            "rep_loss": rep_loss,
            "search_loss": search_loss,
            "abs_mismatch": mismatch,
            "relative_mismatch": (
                mismatch / abs(rep_loss) if abs(rep_loss) > 1e-12 else None
            ),
            "min_valid": pad["min_valid"],
            "queries_lt_100": pad["queries_lt_100"],
            "queries_lt_feedback_k": pad["queries_lt_feedback_k"],
            "queries_lt_evidence_k": pad["queries_lt_evidence_k"],
        })

    cal = pd.DataFrame(rows)
    viable = cal[cal["queries_lt_feedback_k"] == 0].copy()
    assert len(viable) > 0, (
        "No frozen nprobe-grid point returns the required 20 feedback candidates "
        "for every FIT query. STOP; do not modify the grid post hoc."
    )
    viable = viable.sort_values(["abs_mismatch", "nprobe"]).reset_index(drop=True)
    SELECTED_NPROBE = int(viable.iloc[0]["nprobe"])
    cal = cal.sort_values(["abs_mismatch", "nprobe"]).reset_index(drop=True)
    cal.to_csv(CALIBRATION_CSV, index=False)
    selected_row = cal.loc[cal["nprobe"] == SELECTED_NPROBE].iloc[0]

    gate = {
        "study_id": "ARC-v0.37.2.2",
        "selected_nprobe": SELECTED_NPROBE,
        "fit_n": len(FIT_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "fit_rep_loss": float(rep_loss),
        "selected_search_loss": float(selected_row["search_loss"]),
        "abs_mismatch": float(selected_row["abs_mismatch"]),
        "relative_mismatch": (
            float(selected_row["relative_mismatch"])
            if pd.notna(selected_row["relative_mismatch"]) else None
        ),
        "selected_padding_diagnostics": {
            "min_valid": int(selected_row["min_valid"]),
            "queries_lt_100": int(selected_row["queries_lt_100"]),
            "queries_lt_feedback_k": int(selected_row["queries_lt_feedback_k"]),
            "queries_lt_evidence_k": int(selected_row["queries_lt_evidence_k"]),
        },
        "main_retrieval_outcomes_inspected_before_selection": False,
        "valid_retuning": False,
        "tie_break": "smaller nprobe",
    }
    CALIBRATION_GATE.write_text(json.dumps(gate, indent=2), encoding="utf-8")
    CALIBRATION_SHA_PATH.write_text(
        sha256_file(CALIBRATION_GATE) + "\n", encoding="utf-8"
    )

print(pd.read_csv(CALIBRATION_CSV) if CALIBRATION_CSV.is_file() else gate)
print("SELECTED_NPROBE:", SELECTED_NPROBE)
print("Calibration SHA:", sha256_file(CALIBRATION_GATE))
print("FIT CALIBRATION SEALED — PASS")

## Hard boundary

From this point forward, `SELECTED_NPROBE` is frozen. Do not rerun calibration with MAIN
or VALID outcomes, and do not change the search grid after seeing downstream results.

In [ ]:
# Cell 13 — MAIN500 one-shot retrieval and trajectory helpers
MAIN_STARTED = RUN / "MAIN500_RETRIEVAL_STARTED.txt"
if not MAIN_STARTED.is_file():
    MAIN_STARTED.write_text(datetime.now(timezone.utc).isoformat() + "\n", encoding="utf-8")

Q0 = query_matrix(MAIN_IDS)

def feedback_vectors(ids, scores, policy):
    k = int(policy["k"])
    ids = np.asarray(ids, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)
    vc = np.sum(ids >= 0, axis=1)
    assert np.all(vc >= k), (
        f"At least one query has fewer than frozen feedback k={k} real candidates."
    )
    V = gather_vectors(np.asarray(ids[:, :k], dtype=np.int64))
    if policy["family"] == "mean":
        F = V.mean(axis=1)
    else:
        z = np.asarray(scores[:, :k], dtype=np.float64) / float(policy["tau"])
        z -= z.max(axis=1, keepdims=True)
        w = np.exp(z)
        w /= w.sum(axis=1, keepdims=True)
        F = np.einsum("nk,nkd->nd", w.astype(np.float32), V)
    return normalize_rows(F)

def anchored_update(q0, F, alpha):
    return normalize_rows((1.0 - float(alpha)) * q0 + float(alpha) * F)

def jaccard_divergence(a, b):
    A = {int(x) for x in a if int(x) >= 0}
    B = {int(x) for x in b if int(x) >= 0}
    return 1.0 - len(A & B) / len(A | B)

# Round 0 is policy-independent.
D_rep0, I_rep0 = batched_search(pq, Q0, HIGH_NPROBE)
D_high0, I_high0 = batched_search(sq, Q0, HIGH_NPROBE)
D_search0, I_search0 = batched_search(sq, Q0, SELECTED_NPROBE)

for _name, _I in [
    ("rep_low", I_rep0),
    ("high", I_high0),
    ("search_low", I_search0),
]:
    _diag = padding_diagnostics(_I)
    print("MAIN round0 padding", _name, _diag)
    assert _diag["queries_lt_feedback_k"] == 0
    assert _diag["queries_lt_evidence_k"] == 0

U_rep0 = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_rep0)])
U_high0 = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_high0)])
U_search0 = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_search0)])

round0_retrieval = pd.DataFrame({
    "query_id": MAIN_IDS,
    "high_ndcg10": U_high0,
    "rep_low_ndcg10": U_rep0,
    "search_low_ndcg10": U_search0,
})
round0_retrieval["rep_loss"] = round0_retrieval["high_ndcg10"] - round0_retrieval["rep_low_ndcg10"]
round0_retrieval["search_loss"] = round0_retrieval["high_ndcg10"] - round0_retrieval["search_low_ndcg10"]
round0_retrieval.to_csv(RUN / "v0372_main500_round0_retrieval.csv", index=False)

print(round0_retrieval.mean(numeric_only=True))
print("MAIN500 ROUND-0 RETRIEVAL — COMPLETE")

In [ ]:
# Cell 14 — Reconstruct H=4 anchored trajectories; save retrieval bridge + answer evidence rows
TRAJ_PATH = RUN / "v0372_main500_trajectory.parquet"
ROUND0_CAND_PATH = RUN / "v0372_round0_answer_candidates.parquet"
TERM_CAND_PATH = RUN / "v0372_terminal_answer_candidates.parquet"

# Save round-0 top-5 answer evidence rows.
if not ROUND0_CAND_PATH.is_file():
    r0_rows = []
    for i, qid in enumerate(MAIN_IDS):
        for branch, I in [
            ("rep_low", I_rep0),
            ("high", I_high0),
            ("search_low", I_search0),
        ]:
            r0_rows.append({
                "query_id": qid,
                "branch": branch,
                "rows_json": json.dumps(list(map(int, I[i, :EVIDENCE_K]))),
            })
    pd.DataFrame(r0_rows).to_parquet(ROUND0_CAND_PATH, index=False)

trajectory_records = []
terminal_rows = []

for pi, policy in enumerate(POLICIES):
    print(f"[{pi+1}/{len(POLICIES)}] {policy['name']}")
    states = {
        "rep_low": Q0.copy(),
        "high": Q0.copy(),
        "search_low": Q0.copy(),
    }

    for t in range(H + 1):
        D_rep, I_rep = batched_search(pq, states["rep_low"], HIGH_NPROBE)
        D_high, I_high = batched_search(sq, states["high"], HIGH_NPROBE)
        D_search, I_search = batched_search(sq, states["search_low"], SELECTED_NPROBE)

        U_rep = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_rep)])
        U_high = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_high)])
        U_search = np.asarray([ndcg_at_10(r, RELS[q]) for q, r in zip(MAIN_IDS, I_search)])

        for i, qid in enumerate(MAIN_IDS):
            for mechanism, branch, st, I, U in [
                ("representation", "rep_low", states["rep_low"], I_rep, U_rep),
                ("search_effort", "search_low", states["search_low"], I_search, U_search),
            ]:
                signed = float(U_high[i] - U[i])
                trajectory_records.append({
                    "query_id": qid,
                    "policy": policy["name"],
                    "policy_id": pi,
                    "family": policy["family"],
                    "alpha": policy["alpha"],
                    "round": t,
                    "mechanism": mechanism,
                    "semantic": 1.0 - float(np.dot(states["high"][i], st[i])),
                    "candidate": jaccard_divergence(I_high[i], I[i]),
                    "signed": signed,
                    "absolute": abs(signed),
                })

        if t == H:
            for i, qid in enumerate(MAIN_IDS):
                for branch, I in [
                    ("rep_low", I_rep),
                    ("high", I_high),
                    ("search_low", I_search),
                ]:
                    terminal_rows.append({
                        "query_id": qid,
                        "policy": policy["name"],
                        "policy_id": pi,
                        "family": policy["family"],
                        "alpha": policy["alpha"],
                        "branch": branch,
                        "rows_json": json.dumps(list(map(int, I[i, :EVIDENCE_K]))),
                    })
            break

        F_rep = feedback_vectors(I_rep, D_rep, policy)
        F_high = feedback_vectors(I_high, D_high, policy)
        F_search = feedback_vectors(I_search, D_search, policy)

        states["rep_low"] = anchored_update(Q0, F_rep, policy["alpha"])
        states["high"] = anchored_update(Q0, F_high, policy["alpha"])
        states["search_low"] = anchored_update(Q0, F_search, policy["alpha"])

    # Durable partial checkpoint after each policy.
    pd.DataFrame(trajectory_records).to_parquet(
        RUN / "v0372_main500_trajectory_partial.parquet", index=False
    )
    pd.DataFrame(terminal_rows).to_parquet(
        RUN / "v0372_terminal_answer_candidates_partial.parquet", index=False
    )
    gc.collect()

traj = pd.DataFrame(trajectory_records)
terminal_candidates = pd.DataFrame(terminal_rows)

assert len(traj) == N_MAIN_EXPECTED * len(POLICIES) * 2 * (H + 1)
assert len(terminal_candidates) == N_MAIN_EXPECTED * len(POLICIES) * 3

traj.to_parquet(TRAJ_PATH, index=False)
terminal_candidates.to_parquet(TERM_CAND_PATH, index=False)

# Predeclared retrieval-side mechanistic bridge (secondary for v0.37.2).
retrieval_ep = []
for (qid, policy, mech), g in traj.groupby(["query_id", "policy", "mechanism"]):
    g = g.sort_values("round")
    retrieval_ep.append({
        "query_id": qid,
        "policy": policy,
        "mechanism": mech,
        "H1": ols_slope(g["semantic"]),
        "H2": ols_slope(g["candidate"]),
        "H3_abs": ols_slope(g["absolute"]),
        "H3_signed": ols_slope(g["signed"]),
        "terminal_abs": float(g["absolute"].iloc[-1]),
    })

retrieval_ep = pd.DataFrame(retrieval_ep)
retrieval_ep.to_parquet(RUN / "v0372_retrieval_query_policy_endpoints.parquet", index=False)

print("Trajectory rows:", len(traj))
print("Terminal answer candidate rows:", len(terminal_candidates))
print("H=4 TRAJECTORIES — COMPLETE")

In [ ]:
# Cell 15 — Release encoder GPU memory; load pinned deterministic Qwen answerer
del encoder
# Retrieval is complete at this point; release large CPU objects before loading Qwen.
for _name in ["TRAIN", "pq", "sq", "block_arrays"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

def render_answer_prompt(question, evidence):
    passages = []
    for rank, item in enumerate(evidence, 1):
        passages.append(
            f"[{rank}] {item['title']}\n{item['text']}"
        )
    return USER_TEMPLATE.format(
        question=question.strip(),
        passages="\n\n".join(passages),
    )

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    return white_space_fix(
        remove_articles(
            remove_punc(unicodedata.normalize("NFKC", str(s)).lower())
        )
    )

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred, gold):
    from collections import Counter
    p = normalize_answer(pred).split()
    g = normalize_answer(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision = same / len(p)
    recall = same / len(g)
    return 2 * precision * recall / (precision + recall)

tokenizer = AutoTokenizer.from_pretrained(
    ANSWER_MODEL,
    revision=ANSWER_REVISION,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

answer_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
answer_model = AutoModelForCausalLM.from_pretrained(
    ANSWER_MODEL,
    revision=ANSWER_REVISION,
    torch_dtype=answer_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
answer_model.eval()

ANSWER_CACHE_PATH = CACHE / f"v0372_answer_cache_{RUNTIME_PROTOCOL_SHA[:16]}.jsonl"
answer_cache = {}
if ANSWER_CACHE_PATH.is_file():
    with open(ANSWER_CACHE_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line)
                answer_cache[o["key"]] = o["answer"]

def answer_cache_key(question, evidence):
    payload = {
        "answer_model": ANSWER_MODEL,
        "answer_revision": ANSWER_REVISION,
        "prompt_sha256": PROMPT_SHA,
        "question": question,
        "evidence": evidence,
        "do_sample": False,
        "max_new_tokens": ANSWER_MAX_NEW_TOKENS,
    }
    return sha256_text(json.dumps(payload, sort_keys=True, ensure_ascii=False))

def clean_answer(s):
    s = re.sub(r"^```(?:text)?\s*", "", str(s).strip(), flags=re.I)
    s = re.sub(r"\s*```$", "", s)
    s = re.sub(r"^(answer)\s*:\s*", "", s, flags=re.I)
    return " ".join(s.strip().strip('"').strip("'").split())[:256]

@torch.inference_mode()
def answer_batch(items):
    # items: list[(qid, question, evidence)]
    outputs = [None] * len(items)
    missing = []

    for i, (qid, question, evidence) in enumerate(items):
        key = answer_cache_key(question, evidence)
        if key in answer_cache:
            outputs[i] = answer_cache[key]
        else:
            missing.append((i, key, qid, question, evidence))

    for s in range(0, len(missing), ANSWER_BATCH_SIZE):
        batch = missing[s:s+ANSWER_BATCH_SIZE]
        prompts = []
        for _, key, qid, question, evidence in batch:
            user = render_answer_prompt(question, evidence)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
            ]
            prompts.append(
                tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )

        toks = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096,
        )
        device = next(answer_model.parameters()).device
        toks = {k: v.to(device) for k, v in toks.items()}

        generated = answer_model.generate(
            **toks,
            do_sample=False,
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
        )
        input_len = toks["input_ids"].shape[1]
        texts = tokenizer.batch_decode(
            generated[:, input_len:],
            skip_special_tokens=True,
        )

        with ANSWER_CACHE_PATH.open("a", encoding="utf-8") as f:
            for (_, key, qid, question, evidence), txt in zip(batch, texts):
                ans = clean_answer(txt)
                answer_cache[key] = ans
                f.write(json.dumps({
                    "key": key,
                    "answer": ans,
                }, ensure_ascii=False) + "\n")

    for i, (qid, question, evidence) in enumerate(items):
        outputs[i] = answer_cache[answer_cache_key(question, evidence)]
    return outputs

print("Qwen revision:", ANSWER_REVISION)
print("Existing cached answers:", len(answer_cache))
print("ANSWERER LOAD — PASS")

In [ ]:
# Cell 16 — Determinism smoke test before MAIN answer scoring
smoke = pd.read_parquet(ROUND0_CAND_PATH).head(4)
items = []
for r in smoke.itertuples(index=False):
    ev = evidence_for_rows(json.loads(r.rows_json))
    items.append((r.query_id, QUERY_TEXT[r.query_id], ev))

a1 = answer_batch(items)

# Force one exact regeneration per smoke prompt by removing only in-memory entries.
keys = [answer_cache_key(q, ev) for _, q, ev in items]
for key in keys:
    answer_cache.pop(key, None)

a2 = answer_batch(items)
assert a1 == a2, list(zip(a1, a2))

print("DETERMINISTIC ANSWER SMOKE — 100% PASS")

In [ ]:
# Cell 17 — Generate and score 1,500 one-shot answers
ROUND0_ANS_PATH = RUN / "v0372_round0_answers.parquet"
round0_candidates = pd.read_parquet(ROUND0_CAND_PATH)

if ROUND0_ANS_PATH.is_file():
    round0_answers = pd.read_parquet(ROUND0_ANS_PATH)
else:
    items, meta = [], []
    for r in round0_candidates.itertuples(index=False):
        ev = evidence_for_rows(json.loads(r.rows_json))
        items.append((r.query_id, QUERY_TEXT[r.query_id], ev))
        meta.append((r.query_id, r.branch))

    preds = []
    for s in tqdm(range(0, len(items), 64), desc="One-shot answer groups"):
        preds.extend(answer_batch(items[s:s+64]))

    out = []
    for (qid, branch), pred in zip(meta, preds):
        gold = GOLD[qid]
        out.append({
            "query_id": qid,
            "branch": branch,
            "prediction": pred,
            "gold": gold,
            "f1": token_f1(pred, gold),
            "em": exact_match(pred, gold),
        })
    round0_answers = pd.DataFrame(out)
    round0_answers.to_parquet(ROUND0_ANS_PATH, index=False)

assert len(round0_answers) == N_MAIN_EXPECTED * 3
print(round0_answers.groupby("branch")[["f1", "em"]].mean())
print("ROUND-0 ANSWERS — COMPLETE")

In [ ]:
# Cell 18 — Generate and score 12,000 terminal H=4 answers
TERMINAL_ANS_PATH = RUN / "v0372_terminal_answers.parquet"
terminal_candidates = pd.read_parquet(TERM_CAND_PATH)

if TERMINAL_ANS_PATH.is_file():
    terminal_answers = pd.read_parquet(TERMINAL_ANS_PATH)
else:
    items, meta = [], []
    for r in terminal_candidates.itertuples(index=False):
        ev = evidence_for_rows(json.loads(r.rows_json))
        items.append((r.query_id, QUERY_TEXT[r.query_id], ev))
        meta.append((
            r.query_id,
            r.policy,
            int(r.policy_id),
            r.family,
            float(r.alpha),
            r.branch,
        ))

    preds = []
    for s in tqdm(range(0, len(items), 64), desc="Terminal answer groups"):
        preds.extend(answer_batch(items[s:s+64]))

    out = []
    for (qid, policy, policy_id, family, alpha, branch), pred in zip(meta, preds):
        gold = GOLD[qid]
        out.append({
            "query_id": qid,
            "policy": policy,
            "policy_id": policy_id,
            "family": family,
            "alpha": alpha,
            "branch": branch,
            "prediction": pred,
            "gold": gold,
            "f1": token_f1(pred, gold),
            "em": exact_match(pred, gold),
        })
    terminal_answers = pd.DataFrame(out)
    terminal_answers.to_parquet(TERMINAL_ANS_PATH, index=False)

assert len(terminal_answers) == N_MAIN_EXPECTED * len(POLICIES) * 3
print(terminal_answers.groupby("branch")[["f1", "em"]].mean())
print("TERMINAL ANSWERS — COMPLETE")

In [ ]:
# Cell 19 — Frozen query-level primary + secondary inference
def paired_bootstrap(v, reps=BOOTSTRAP_REPS, seed=SEED):
    v = np.asarray(v, dtype=np.float64)
    assert np.isfinite(v).all()
    rng = np.random.default_rng(seed)
    n = len(v)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 500
    pos = 0
    while pos < reps:
        m = min(chunk, reps - pos)
        idx = rng.integers(0, n, size=(m, n))
        boots[pos:pos+m] = v[idx].mean(axis=1)
        pos += m
    return {
        "n": int(n),
        "mean": float(v.mean()),
        "ci95": [
            float(np.quantile(boots, 0.025)),
            float(np.quantile(boots, 0.975)),
        ],
    }

r0 = round0_answers.pivot(index="query_id", columns="branch", values=["f1", "em"])

term_q = (
    terminal_answers
    .groupby(["query_id", "branch"], as_index=False)[["f1", "em"]]
    .mean()
)
th = term_q.pivot(index="query_id", columns="branch", values=["f1", "em"])

common = sorted(set(r0.index) & set(th.index))
assert len(common) == N_MAIN_EXPECTED
assert set(common) == set(MAIN_IDS)

D0_f1 = (
    r0.loc[common, ("f1", "search_low")].to_numpy()
    - r0.loc[common, ("f1", "rep_low")].to_numpy()
)
DH_f1 = (
    th.loc[common, ("f1", "search_low")].to_numpy()
    - th.loc[common, ("f1", "rep_low")].to_numpy()
)
delta_f1 = DH_f1 - D0_f1

D0_em = (
    r0.loc[common, ("em", "search_low")].to_numpy()
    - r0.loc[common, ("em", "rep_low")].to_numpy()
)
DH_em = (
    th.loc[common, ("em", "search_low")].to_numpy()
    - th.loc[common, ("em", "rep_low")].to_numpy()
)
delta_em = DH_em - D0_em

primary = paired_bootstrap(delta_f1, seed=SEED + 1)
lo, hi = primary["ci95"]
primary["classification"] = (
    "POSITIVE_FEEDBACK_ADDED_QA_CONSEQUENCE" if lo > 0 else
    "REVERSED_FEEDBACK_ADDED_QA_CONSEQUENCE" if hi < 0 else
    "UNRESOLVED"
)

secondary = {
    "terminal_F1_search_minus_rep": paired_bootstrap(DH_f1, seed=SEED + 2),
    "oneshot_F1_search_minus_rep": paired_bootstrap(D0_f1, seed=SEED + 3),
    "feedback_added_EM_contrast": paired_bootstrap(delta_em, seed=SEED + 4),
    "terminal_EM_search_minus_rep": paired_bootstrap(DH_em, seed=SEED + 5),
    "oneshot_EM_search_minus_rep": paired_bootstrap(D0_em, seed=SEED + 6),
    "terminal_high_minus_rep_F1": paired_bootstrap(
        th.loc[common, ("f1", "high")].to_numpy()
        - th.loc[common, ("f1", "rep_low")].to_numpy(),
        seed=SEED + 7,
    ),
    "terminal_high_minus_search_F1": paired_bootstrap(
        th.loc[common, ("f1", "high")].to_numpy()
        - th.loc[common, ("f1", "search_low")].to_numpy(),
        seed=SEED + 8,
    ),
}

# Retrieval-side H3_abs bridge, averaged over policies within query.
rep_ep = retrieval_ep[retrieval_ep["mechanism"] == "representation"]
sea_ep = retrieval_ep[retrieval_ep["mechanism"] == "search_effort"]
rep_q = rep_ep.groupby("query_id")["H3_abs"].mean()
sea_q = sea_ep.groupby("query_id")["H3_abs"].mean()
retrieval_bridge = paired_bootstrap(
    rep_q.loc[common].to_numpy() - sea_q.loc[common].to_numpy(),
    seed=SEED + 20,
)

# Answer disagreement to shared high.
def norm_pred(s):
    return normalize_answer(s)

r0_pred = round0_answers.pivot(index="query_id", columns="branch", values="prediction")
round0_disagreement = {
    "rep_vs_high": float(np.mean([
        norm_pred(r0_pred.loc[q, "rep_low"]) != norm_pred(r0_pred.loc[q, "high"])
        for q in common
    ])),
    "search_vs_high": float(np.mean([
        norm_pred(r0_pred.loc[q, "search_low"]) != norm_pred(r0_pred.loc[q, "high"])
        for q in common
    ])),
}

term_pred = terminal_answers.copy()
term_pred["prediction_norm"] = term_pred["prediction"].map(norm_pred)
merged_rep = term_pred[term_pred.branch == "rep_low"].merge(
    term_pred[term_pred.branch == "high"],
    on=["query_id", "policy"],
    suffixes=("_rep", "_high"),
)
merged_search = term_pred[term_pred.branch == "search_low"].merge(
    term_pred[term_pred.branch == "high"],
    on=["query_id", "policy"],
    suffixes=("_search", "_high"),
)
terminal_disagreement = {
    "rep_vs_high_query_policy": float(np.mean(
        merged_rep["prediction_norm_rep"] != merged_rep["prediction_norm_high"]
    )),
    "search_vs_high_query_policy": float(np.mean(
        merged_search["prediction_norm_search"] != merged_search["prediction_norm_high"]
    )),
}

query_out = pd.DataFrame({
    "query_id": common,
    "D0_f1_search_minus_rep": D0_f1,
    "DH_f1_search_minus_rep": DH_f1,
    "feedback_added_f1_contrast": delta_f1,
    "D0_em_search_minus_rep": D0_em,
    "DH_em_search_minus_rep": DH_em,
    "feedback_added_em_contrast": delta_em,
})
query_out.to_csv(RUN / "v0372_query_level_answer_endpoints.csv", index=False)

result = {
    "study_id": "ARC-v0.37.2.2",
    "source_v0371_protocol_sha256": SOURCE_V0371_PROTOCOL_SHA,
    "runtime_protocol_sha256": RUNTIME_PROTOCOL_SHA,
    "calibration_sha256": sha256_file(CALIBRATION_GATE),
    "selected_nprobe": SELECTED_NPROBE,
    "n_main": N_MAIN_EXPECTED,
    "primary": primary,
    "secondary": secondary,
    "retrieval_bridge_secondary": retrieval_bridge,
    "descriptive": {
        "round0_branch_mean": (
            round0_answers.groupby("branch")[["f1", "em"]].mean().to_dict()
        ),
        "terminal_branch_mean": (
            terminal_answers.groupby("branch")[["f1", "em"]].mean().to_dict()
        ),
        "round0_answer_disagreement_to_high": round0_disagreement,
        "terminal_answer_disagreement_to_high": terminal_disagreement,
        "fraction_queries_primary_positive": float(np.mean(delta_f1 > 0)),
        "fraction_queries_primary_negative": float(np.mean(delta_f1 < 0)),
        "fraction_queries_primary_tied": float(np.mean(delta_f1 == 0)),
    },
    "retuned_after_main_outcomes": False,
    "retuned_after_answer_outcomes": False,
}
PRIMARY_GATE_PATH = RUN / "v0372_primary_gate.json"
PRIMARY_GATE_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")

print(json.dumps(primary, indent=2))
print("Retrieval H3_abs bridge:", json.dumps(retrieval_bridge, indent=2))
print("FROZEN PRIMARY INFERENCE — COMPLETE")

In [ ]:
# Cell 20 — Conservative manuscript wording + artifact SHA manifest
p = result["primary"]
lo, hi = p["ci95"]

if p["classification"] == "POSITIVE_FEEDBACK_ADDED_QA_CONSEQUENCE":
    wording = (
        "On a prospectively frozen HotpotQA native-gold benchmark, feedback adds a "
        "downstream answer-quality separation between representation approximation and "
        "matched search-effort approximation beyond their one-shot answer difference: "
        f"Delta_QA={p['mean']:+.4f} token-F1, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "The claim is limited to this frozen 500-query HotpotQA subset, GTE-small, "
        "anchored H=4, the eight prespecified policies, and Qwen2.5-3B-Instruct."
    )
elif p["classification"] == "REVERSED_FEEDBACK_ADDED_QA_CONSEQUENCE":
    wording = (
        "The prospectively frozen HotpotQA downstream audit reverses the signed "
        "feedback-added answer-quality ordering: "
        f"Delta_QA={p['mean']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "This is an answer-level estimand boundary and should be retained without retuning."
    )
else:
    wording = (
        "The prospectively frozen HotpotQA downstream audit is unresolved for the signed "
        "feedback-added answer-quality contrast: "
        f"Delta_QA={p['mean']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "The retrieval-trajectory result therefore should not be promoted to a signed "
        "answer-quality claim under this frozen answerer and evidence contract."
    )

final_report = {
    "study_id": "ARC-v0.37.2.2",
    "status": "COMPLETE",
    "primary": p,
    "retrieval_bridge_secondary": result["retrieval_bridge_secondary"],
    "suggested_manuscript_wording": wording,
    "claim_guardrail": (
        "Independent native-gold benchmark freeze occurred before retrieval calibration "
        "and before answer outcomes. Retain positive/null/reversed result; do not retune."
    ),
}
(RUN / "v0372_final_report.json").write_text(
    json.dumps(final_report, indent=2), encoding="utf-8"
)

# Compact artifact manifest: hashes all run files but intentionally not 4GB embedding
# blocks or multi-GB indexes (those are covered by their manifests/hashes).
artifact_rows = []
for path in sorted(RUN.iterdir()):
    if path.is_file() and path.name != "V0372_ARTIFACT_SHA256.csv":
        artifact_rows.append({
            "file": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
pd.DataFrame(artifact_rows).to_csv(
    RUN / "V0372_ARTIFACT_SHA256.csv", index=False
)

print(wording)
print("\nRun artifacts:", RUN)
print("ARC-v0.37.2.2 — COMPLETE")

## Reporting guardrail

Do not change the answer model, prompt, evidence depth, selected nprobe, MAIN500
membership, H, policies, metric, or primary after seeing the result.

- **Positive CI:** one prospectively frozen downstream QA consequence supports the
  paper's practical-consequence story.
- **CI crosses zero:** retain the null; trajectory non-equivalence does not establish
  signed answer-quality transfer under this setting.
- **Negative CI:** retain the reversal as an estimand/operator boundary.

This audit remains one HotpotQA/GTE-small/Qwen setting. It does not establish a universal
RAG answer-quality law.